In [ ]:


#     _UNITLESS = {"r2"}
#     _PCT      = {"mape", "wmape", "spike_pct", "dip_pct"}

#     m = eval_output["model"]
#     n = eval_output["naive"]
#     for metric, val in m.items():
#         if metric in _UNITLESS:
#             fmt       = f"{val:8.4f}"
#             naive_fmt = f"{n.get(metric, 0.0):8.4f}"
#         elif metric in _PCT:
#             unit      = "%"
#             fmt       = f"{val:8.2f}{unit}"
#             naive_fmt = f"{n.get(metric, 0.0):8.2f}{unit}" if metric in n else "        n/a"
#         elif metric in ("spike_mae", "nonspike_mae"):
#             unit      = " $/MWh"
#             fmt       = f"{val:8.2f}{unit}"
#             naive_fmt = "        n/a"
#         else:
#             unit      = " $/MWh"
#             fmt       = f"{val:8.2f}{unit}"
#             naive_fmt = f"{n.get(metric, 0.0):8.2f}{unit}"
#         print(f"  {metric.upper():14s}  LGBM: {fmt}   Naive lag-48: {naive_fmt}", flush=True)

#     naive_mae = n.get("mae", 0.0)
#     if naive_mae > 0:
#         skill = (1 - m["mae"] / naive_mae) * 100
#         print(f"  {'SKILL':14s}  MAE skill vs naive: {skill:+.2f}%", flush=True)

#     print(
#         f"\n  ── Per-step Metrics "
#         f"(gap={FORECAST_GAP * INTERVAL_MINUTES // 60}h, "
#         f"horizon={FORECAST_HORIZON * INTERVAL_MINUTES // 60}h) ─────────────────",
#         flush=True,
#     )
#     print(f"    {'step':>4}  {'lead':>6}  {'MAE':>8}  {'RMSE':>8}  {'R²':>7}  {'MBE':>8}", flush=True)
#     for _, row in steps_df.iterrows():
#         print(
#             f"    step {int(row['step']):>3}  "
#             f"lead={row['lead_h']:.1f}h  "
#             f"MAE={row['mae']:>7.2f}  "
#             f"RMSE={row['rmse']:>7.2f}  "
#             f"R²={row['r2']:>6.4f}  "
#             f"MBE={row['mbe']:>+7.2f}",
#             flush=True,
#         )

#     fi_df = eval_output.get("feature_importance")
#     if fi_df is not None:
#         print(f"\n  Top 20 features by mean gain (base models)", flush=True)
#         for rank, row in fi_df.head(20).iterrows():
#             print(f"    {rank + 1:>3}.  {row['feature']:<45}  {row['importance']:>12.0f}")




        model = joblib.load(model_file)
        
        stored_feature_cols = model.get("feature_cols", feature_cols)
        scaler = model.get("scaler", None)
        eval_output = evaluate_seq2seq(model, df_full, stored_feature_cols, scaler)
        
        print(f"  [+] MAE: {eval_output['model']['mae']:7.2f} $/MWh", flush=True)
        if 'spike_pct' in eval_output['model']:
            print(f"  [+] Spikes: {eval_output['model']['spike_pct']:5.2f}%", flush=True)
        print(f"  [OK] {region} complete", flush=True)
        
        return {
            "state": region,
            "region_tag": region_tag,
            "eval_output": eval_output,
            "model": model,
        }
    
  


            # Display key metrics
            key_metrics = ["state", "mae", "rmse", "r2", "mbe", "wmape"]
            if "spike_pct" in metrics_df.columns:
                key_metrics.extend(["spike_mae", "spike_pct", "dip_mae"])
        

            # Aggregate metrics
            m = eval_out["model"]

            for key, val in m.items():
                if key in ("spike_pct", "dip_pct", "wmape"):
                    f.write(f"  {key:20s}: {val:8.2f}%\n")
                elif key in ("r2",):
                    f.write(f"  {key:20s}: {val:8.4f}\n")
                else:
                    f.write(f"  {key:20s}: {val:8.2f} $/MWh\n")
            
            # Naive comparison
          
            n = eval_out["naive"]
            for key in ["mae", "rmse", "r2", "mbe"]:
                if key in n:
                    if key == "r2":
                        f.write(f"  {key:20s}: {n[key]:8.4f}\n")
                    else:
                        f.write(f"  {key:20s}: {n[key]:8.2f} $/MWh\n")
            
            # Skill score
            if n.get("mae", 0) > 0:
                skill = (1 - m["mae"] / n["mae"]) * 100
                f.write(f"  {'SKILL SCORE':20s}: {skill:+.2f}%\n")
            
            # Per-step metrics
            steps_df = eval_out["steps_df"]
            f.write(f"{'Step':>6} {'Lead':>6} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'MBE':>8}\n")
            f.write("-" * 80 + "\n")
            for _, row in steps_df.iterrows():
                f.write(
                    f"{int(row['step']):>6} "
                    f"{row['lead_h']:>6.1f}h "
                    f"{row['mae']:>8.2f} "
                    f"{row['rmse']:>8.2f} "
                    f"{row['r2']:>8.4f} "
                    f"{row['mbe']:>8.2f}\n"
                )
            
            # Feature importance
            fi_df = eval_out.get("feature_importance")
            if fi_df is not None and not fi_df.empty:
                f.write("\nTOP 20 MOST IMPORTANT FEATURES\n")
                f.write("-" * 80 + "\n")
                for idx, (_, row) in enumerate(fi_df.head(20).iterrows(), start=1):
                    f.write(f"  {idx:2d}. {row['feature']:40s} {row['importance']:10.6f}\n")
            
       
